# Input Corpus Shuffle
Fast preprocessing shuffle for `input-images/*` corpuses (v4).

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        if (current / 'input-images').exists() and (current / 'rb_pipeline_v4').exists():
            return current

    raise RuntimeError('Could not locate repository root with input-images/ and rb_pipeline_v4/.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/mitch/development/raccoon-ball/02_synthetic-data-processing-v4.0')

In [2]:
import html

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

from rb_pipeline_v4.input_corpus_shuffle import (
    default_input_shuffle_destination,
    discover_input_corpuses,
    parse_shuffle_seed,
    shuffle_input_corpus,
)

INPUT_ROOT = REPO_ROOT / 'input-images'
_summaries = []

title = widgets.HTML('<b>Input Corpus Shuffle (Preprocessing)</b>')
summary_html = widgets.HTML()
seed_error_html = widgets.HTML()

corpus_dropdown = widgets.Dropdown(
    description='Corpus:',
    options=[],
    layout=widgets.Layout(width='98%'),
)
seed_text = widgets.Text(
    description='Seed:',
    value='',
    placeholder='Required integer seed',
)
refresh_button = widgets.Button(description='Refresh Corpuses')
execute_button = widgets.Button(
    description='Execute Shuffle',
    button_style='primary',
    disabled=True,
)
status_output = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='8px'))


def _seed_error() -> str | None:
    try:
        parse_shuffle_seed(seed_text.value)
    except Exception as exc:
        return str(exc)
    return None


def _selected_corpus_path() -> Path | None:
    value = corpus_dropdown.value
    if not value:
        return None
    return Path(value)


def _render_summary_table(summaries) -> str:
    if not summaries:
        return '<i>No corpuses found under input-images.</i>'

    header = (
        '<tr>'
        '<th align="left">Corpus</th>'
        '<th align="right">Rows</th>'
        '<th align="right">Images found</th>'
        '<th align="right">Images missing</th>'
        '<th align="left">Status</th>'
        '</tr>'
    )
    rows = []
    for summary in summaries:
        name = html.escape(summary.name)
        rows_count = '' if summary.samples_row_count is None else str(summary.samples_row_count)
        found = '' if summary.referenced_images_found is None else str(summary.referenced_images_found)
        missing = '' if summary.referenced_images_missing is None else str(summary.referenced_images_missing)
        status = html.escape(summary.validation_message)
        tone = '#2e7d32' if summary.validates_cleanly else '#b71c1c'
        rows.append(
            '<tr>'
            f'<td>{name}</td>'
            f'<td align="right">{rows_count}</td>'
            f'<td align="right">{found}</td>'
            f'<td align="right">{missing}</td>'
            f'<td style="color:{tone}">{status}</td>'
            '</tr>'
        )

    return (
        '<table style="border-collapse:collapse; width:100%;">'
        f'{header}'
        f"{''.join(rows)}"
        '</table>'
    )


def _refresh_execute_state(*_args) -> None:
    seed_error = _seed_error()
    selected_path = _selected_corpus_path()
    executable = (seed_error is None) and (selected_path is not None)
    execute_button.disabled = not executable

    if seed_error is None:
        seed_error_html.value = ''
    else:
        seed_error_html.value = f'<span style="color:#b71c1c">{html.escape(seed_error)}</span>'


def _refresh_corpuses(*_args) -> None:
    global _summaries
    _summaries = discover_input_corpuses(INPUT_ROOT)
    selectable = [s for s in _summaries if s.selectable]

    options = []
    for summary in selectable:
        label = summary.name
        if not summary.validates_cleanly:
            label = f'{label} (validation issues)'
        options.append((label, str(summary.path)))

    corpus_dropdown.options = options
    if options and corpus_dropdown.value not in {value for _, value in options}:
        corpus_dropdown.value = options[0][1]

    summary_html.value = _render_summary_table(_summaries)
    _refresh_execute_state()


def _on_execute(_button) -> None:
    seed_value = parse_shuffle_seed(seed_text.value)
    selected = _selected_corpus_path()
    if selected is None:
        return

    execute_button.disabled = True
    refresh_button.disabled = True
    seed_text.disabled = True
    corpus_dropdown.disabled = True

    with status_output:
        clear_output(wait=True)
        print('Starting input corpus shuffle...')
        print(f'  Source: {selected}')
        print(f'  Seed:   {seed_value}')
        print(f'  Target: {default_input_shuffle_destination(selected)}')

    try:
        result = shuffle_input_corpus(selected, seed_value)
    except Exception as exc:
        with status_output:
            print(f'\nError: {exc}')
    else:
        with status_output:
            print('')
            print('Completed successfully.')
            print(f'  Output rows: {result.output_row_count}')
            if result.excluded_capture_failed_rows:
                print(f'  Excluded capture_success=false rows: {result.excluded_capture_failed_rows}')
            print(f'  Destination: {result.destination_corpus_path}')
        _refresh_corpuses()
    finally:
        refresh_button.disabled = False
        seed_text.disabled = False
        corpus_dropdown.disabled = False
        _refresh_execute_state()


refresh_button.on_click(_refresh_corpuses)
execute_button.on_click(_on_execute)
seed_text.observe(_refresh_execute_state, names='value')
corpus_dropdown.observe(_refresh_execute_state, names='value')

_refresh_corpuses()

display(
    widgets.VBox(
        [
            title,
            widgets.HBox([refresh_button]),
            corpus_dropdown,
            seed_text,
            seed_error_html,
            execute_button,
            widgets.HTML('<b>Corpus Summary</b>'),
            summary_html,
            widgets.HTML('<b>Status</b>'),
            status_output,
        ]
    )
)